# Stage 2: Lin-KK Quality

Validates each selected spectrum for Kramers-Kronig compliance and picks the best replica per gas condition and temperature.

**Reads:** `{sample_id}/Results/{condition}/stage1_labeling.xlsx` · `{sample_id}/ISM validation/*.ism`
**Writes:** `{sample_id}/Results/{condition}/stage2_kk.xlsx`

Set `FOCUS_T` to process one temperature at a time. The export is merge-aware: other temperatures are preserved in `stage2_kk.xlsx`.

## Quick links

Steps below are cells inside this notebook, not pipeline Stages (this whole notebook is Stage 2).

- [Configuration](#configuration): sample_id, KK parameters, KK_OVERRIDES, OVERRIDES
- [Step 1: Batch KK](#step-1-batch-kk): silent run and GREEN/YELLOW/RED classification table
- [Step 2: Tune flagged spectra](#step-2-tune-flagged-spectra): interactive panel for YELLOW/RED
- [Step 3: Export](#step-3-selection-summary-and-export): selection summary and stage2_kk.xlsx export

## Configuration

**Mode switch** `PARAM_MODE` (top of the cell below), hand-edited and the cell re-run to switch:

- `"lock"`: read-only reproduction. Everything loads from `session.json`, nothing writes back.
- `"continue"`: starting values load from `session.json` when present; widget/Apply edits merge-save back. Use this to keep tuning a calibration across sessions.
- `"reset"`: ignores `session.json`, starts from the notebook's own default values, and the next save **overwrites** the saved calibration on purpose. Use only when you deliberately want to start this sample's Lin-KK tuning over.

`reset` only affects the scalar KK settings (`KK_C`, IQR fence/window, f_min/f_max, W-criteria). Per-condition overrides from the tuning panel (Step 2) are never touched by this switch.

In [ ]:
from pathlib import Path
from pipeline.interactive import select_sample, discover_conditions, discover_conditions_from_session, param_source_banner
from pipeline.session import LOCKED_MSG as _LOCKED_MSG, load_sample, update_sample

NOTEBOOK_DIR = Path.cwd()

sample_id = select_sample(NOTEBOOK_DIR, show_list=True)

_cfg = load_sample(sample_id)

sample_dir     = NOTEBOOK_DIR / sample_id
all_conditions = discover_conditions(sample_dir) or discover_conditions_from_session(_cfg)

# Order conditions by representative p(O2) (high to low) for the selector
# and the ordered session save; no-p(O2) conditions trail alphabetically.
from pipeline.utils import condition_pO2_map
from pipeline.interactive import order_conditions_by_pO2
_pO2_map = condition_pO2_map(sample_dir, all_conditions)
all_conditions = order_conditions_by_pO2(all_conditions, _pO2_map)

FOCUS_T         = None
SKIP_EXISTING   = False

# lock: read-only, no write.
# continue: read session.json if present, edits merge-save.
# reset: scalars only (KK_C, KK_USE_W_CRITERIA, IQR fence/window, f_min/f_max
#        hard). Ignores session.json, starts from literals below, next save
#        overwrites. KK_OVERRIDES/OVERRIDES untouched.
# Full semantics: README "Changing parameters" + Configuration markdown above.
PARAM_MODE = "continue"

# Starting values load from session.json unless reset (same rule as
# KK_OVERRIDES/OVERRIDES below): re-run must never silently wipe saved tuning.
_p2 = {} if PARAM_MODE == "reset" else _cfg.get("stage2_params", {})
param_source_banner(PARAM_MODE, "Stage 2")
# M selection follows the RelaxIS manual (sect. 7.1.2): False = "Percentage" mode
# with M = round(KK_C*N); True = automatic mu >= KK_MU_TARGET (Schoenleber 2014).
# KK_C = 0.76 is calibrated on this dataset, not a RelaxIS default (View default
# is 50%): same mean kk_score as 50% but keeps more HF window before the IQR cut
# (evidence: kk_mode_comparison.csv in the sample's local calibration folder).
KK_USE_BINARY_M   = _p2.get("KK_USE_BINARY_M",   False)
KK_MU_TARGET      = _p2.get("KK_MU_TARGET",       0.50)
KK_C              = _p2.get("KK_C",               0.76)
KK_IQR_FENCE      = _p2.get("KK_IQR_FENCE",       2.0)
KK_IQR_WINDOW     = _p2.get("KK_IQR_WINDOW",      5)
# None = adaptive only; production runs set a hard LF floor (e.g. 80 Hz) via saved params.
KK_F_MIN_HARD     = _p2.get("KK_F_MIN_HARD",      None)
KK_F_MAX_HARD     = _p2.get("KK_F_MAX_HARD",      None)
# W_re / W_im are Shapiro-Wilk statistics on the |Z|-normalized Lin-KK residuals:
# gaussian residuals (W close to 1) mean pure noise, i.e. no systematic KK violation.
# False -> strict single metric: kk_score = (W_re + W_im)/2 >= 0.97, best for clean spectra.
# True  -> ceramic dual criterion: W_re >= 0.95 AND W_im >= 0.93 AND freq_cut <= 20%;
#          looser on Z_im because high-impedance ceramics have noisier imaginary parts.
#          Enable only when the strict score rejects spectra that look valid by eye.
KK_USE_W_CRITERIA = _p2.get("KK_USE_W_CRITERIA",  False)

# Widget-written stores: always load from session.json, re-run never drops them.
# reset never touches these, only stage2_params.
KK_OVERRIDES = {cond: {int(t): v for t, v in td.items()}
                for cond, td in _cfg.get("kk_overrides", {}).items()}
OVERRIDES    = {cond: {int(t): v for t, v in td.items()}
                for cond, td in _cfg.get("overrides", {}).items()}


def _stage2_params() -> dict:
    return {
        "KK_USE_BINARY_M":   KK_USE_BINARY_M,
        "KK_MU_TARGET":      KK_MU_TARGET,
        "KK_C":              KK_C,
        "KK_IQR_FENCE":      KK_IQR_FENCE,
        "KK_IQR_WINDOW":     KK_IQR_WINDOW,
        "KK_F_MIN_HARD":     KK_F_MIN_HARD,
        "KK_F_MAX_HARD":     KK_F_MAX_HARD,
        "KK_USE_W_CRITERIA": KK_USE_W_CRITERIA,
    }

def _save_params() -> bool:
    """Save only stage2_params. No-op (returns False) in lock mode."""
    if PARAM_MODE == "lock":
        return False
    update_sample(sample_id, stage2_params=_stage2_params())
    return True

def _save_overrides() -> bool:
    """Merge-save kk_overrides, overrides and stage2_params.
    No-op (returns False) in lock mode."""
    if PARAM_MODE == "lock":
        return False
    update_sample(sample_id,
                  conditions=all_conditions,
                  stage2_params=_stage2_params(),
                  kk_overrides=KK_OVERRIDES,
                  overrides=OVERRIDES)
    return True

_ = _save_overrides()

In [ ]:
from pipeline.interactive import make_condition_selector

def _set_focus_t(T):
    global FOCUS_T
    FOCUS_T = T

# Selection only: pick conditions and (optionally) one temperature, then run
# Step 1, which processes exactly this selection.
get_selected_conditions = make_condition_selector(
    all_conditions,
    temps=[600, 575, 550, 525, 500, 475, 450, 425, 400],
    set_focus_t=_set_focus_t,
    pO2_map=_pO2_map,
)

In [ ]:
import ipywidgets as W
from IPython.display import display, HTML

# What this cell configures: the Lin-KK validity test (Schoenleber 2014).
# The test rebuilds each spectrum from M RC elements; if the reconstruction
# residuals are pure noise, the spectrum is causal/linear/stable (KK valid).
display(HTML(
    "<div style='font-size:12px;color:#444;max-width:760px;line-height:1.5'>"
    "<b>Lin-KK parameters.</b> <code>KK_C</code> sets how many RC elements the test "
    "uses (M = KK_C*N): fewer elements means a stricter (more rigid) reconstruction, "
    "more elements lets the fit absorb real KK violations into noise. "
    "<code>f_min</code>/<code>f_max</code> are hard cutoffs applied before testing: "
    "these reject instrumental artefacts (LF drift, electrode polarization, HF cable "
    "inductance), so they are measurement-setup-dependent, not a test-statistic knob. "
    "The IQR fence and window then trim edge points whose residuals are outliers "
    "(fence = distance from the interquartile range counted as outlier; window = "
    "consecutive clean points needed to confirm the edge): a residual-cleanup "
    "convention calibrated once for this dataset (2.0/5), not something presets vary. "
    "<b>W-criteria:</b> W_re/W_im are Shapiro-Wilk statistics of the residuals; "
    "W close to 1 means the residuals are pure noise, i.e. no systematic KK violation. "
    "The strict criterion requires (W_re+W_im)/2 &ge; 0.97. The ceramic criterion "
    "requires W_re &ge; 0.95 AND W_im &ge; 0.93, looser on Z_im because "
    "high-impedance ceramics have noisier imaginary parts. Enable it only when the "
    "strict score rejects spectra that look valid by eye.</div>"))

_w_fmin = W.BoundedFloatText(
    value=KK_F_MIN_HARD or 50, min=0.001, max=1e6, step=1,
    description="f_min [Hz]:", style={"description_width": "110px"}, layout=W.Layout(width="260px"))
_w_fmax_none = W.Checkbox(
    value=KK_F_MAX_HARD is None, description="No limit",
    layout=W.Layout(width="110px"))
_w_fmax = W.BoundedFloatText(
    value=KK_F_MAX_HARD or 5e6, min=1, max=1e8, step=1000,
    description="f_max [Hz]:", disabled=(KK_F_MAX_HARD is None),
    style={"description_width": "110px"}, layout=W.Layout(width="260px"))
_w_c = W.BoundedFloatText(
    value=KK_C, min=0.1, max=1.0, step=0.01,
    description="KK_C:", style={"description_width": "110px"}, layout=W.Layout(width="260px"))
_w_fence = W.BoundedFloatText(
    value=KK_IQR_FENCE, min=0.1, max=5.0, step=0.1,
    description="IQR fence:", style={"description_width": "110px"}, layout=W.Layout(width="260px"))
_w_window = W.BoundedIntText(
    value=KK_IQR_WINDOW, min=3, max=20, step=1,
    description="IQR window:", style={"description_width": "110px"}, layout=W.Layout(width="260px"))
_w_wcrit = W.Checkbox(value=KK_USE_W_CRITERIA, description="Use W-criteria (ceramic electrolyte)")
_w_status = W.HTML()

_wp_suspend = [False]

def _on_param_change(_change=None):
    global KK_F_MIN_HARD, KK_F_MAX_HARD, KK_C, KK_IQR_FENCE, KK_IQR_WINDOW, KK_USE_W_CRITERIA
    if _wp_suspend[0]:
        return
    if PARAM_MODE == "lock":
        _w_status.value = f"<span style='color:#9a6700;font-size:11px'>{_LOCKED_MSG}</span>"
        return
    KK_F_MIN_HARD     = _w_fmin.value
    KK_F_MAX_HARD     = None if _w_fmax_none.value else _w_fmax.value
    KK_C              = _w_c.value
    KK_IQR_FENCE      = _w_fence.value
    KK_IQR_WINDOW     = _w_window.value
    KK_USE_W_CRITERIA = _w_wcrit.value
    _w_fmax.disabled  = _w_fmax_none.value
    _save_params()
    _w_status.value = "<span style='color:green;font-size:11px'>saved</span>"

for _w in (_w_fmin, _w_fmax_none, _w_fmax, _w_c, _w_fence, _w_window, _w_wcrit):
    _w.observe(_on_param_change, names="value")

# Presets set only KK_C (test strictness) and KK_USE_W_CRITERIA (acceptance rule):
# the two parameters that are genuine statistical/methodological choices about the
# Lin-KK test itself. f_min/f_max (instrumental artefact cutoffs, sample-dependent)
# and IQR fence/window (residual-outlier convention, fixed dataset-wide) are left
# exactly as set in the panel above; presets never touch them.
KK_PRESETS = {
    "Conservative (publication)": dict(KK_C=0.85, KK_USE_W_CRITERIA=False),
    "Standard (current optimum)": dict(KK_C=0.76, KK_USE_W_CRITERIA=False),
    "Permissive (ceramic electrolyte-aware)": dict(KK_C=0.76, KK_USE_W_CRITERIA=True),
    "RelaxIS defaults": dict(KK_C=0.85, KK_USE_W_CRITERIA=False),
}
_PRESET_WHY = {
    "Conservative (publication)":
        "Strictest reconstruction: KK_C 0.85 (fewer RC elements, less room to "
        "absorb real violations). Use for the final, publication-grade validation.",
    "Standard (current optimum)":
        "Dataset-calibrated default: KK_C 0.76, same mean acceptance as "
        "Conservative but a more flexible reconstruction.",
    "Permissive (ceramic electrolyte-aware)":
        "Same KK_C as Standard, but switches to W-criteria acceptance: for "
        "high-impedance ceramics whose noisy Z_im fails the strict score "
        "even when the spectrum looks valid by eye.",
    "RelaxIS defaults":
        "Manufacturer default KK_C (0.85). Use to cross-check this pipeline "
        "against a RelaxIS analysis of the same data.",
}
_wp = W.Dropdown(options=list(KK_PRESETS), value="Standard (current optimum)",
                 description="Preset:", layout=W.Layout(width="380px"))
_wp_btn = W.Button(description="Apply preset", button_style="warning",
                   layout=W.Layout(width="140px"),
                   tooltip="Set KK_C and KK_USE_W_CRITERIA to this preset and save "
                           "(f_min/f_max and IQR fence/window are untouched)")
_wp_why = W.HTML()
_wp_msg = W.HTML()

def _show_why(*_):
    _wp_why.value = ("<span style='font-size:12px;color:#555'>"
                     + _PRESET_WHY[_wp.value] + "</span>")
_wp.observe(_show_why, names="value")
_show_why()

def _apply_preset(_b):
    global KK_C, KK_USE_W_CRITERIA
    if PARAM_MODE == "lock":
        _wp_msg.value = f"<b style='color:#9a6700'>{_LOCKED_MSG}</b>"
        return
    _p = KK_PRESETS[_wp.value]
    KK_C              = _p["KK_C"]
    KK_USE_W_CRITERIA = _p["KK_USE_W_CRITERIA"]
    _wp_suspend[0] = True
    try:
        # mirror into the widgets without re-triggering the save observer
        _w_c.value     = KK_C
        _w_wcrit.value = KK_USE_W_CRITERIA
    finally:
        _wp_suspend[0] = False
    _save_params()
    _wp_msg.value = (f"<b style='color:#b36b00'>Applied '{_wp.value}' and saved.</b> "
                     "Re-run Step 1 to reclassify.")
_wp_btn.on_click(_apply_preset)

display(W.VBox([
    W.Label("KK parameters:"),
    _w_fmin,
    W.HBox([_w_fmax, _w_fmax_none]),
    _w_c,
    _w_fence,
    _w_window,
    _w_wcrit,
    _w_status,
    W.HBox([_wp, _wp_btn]),
    _wp_why,
    _wp_msg,
]))

## Import

In [ ]:
import sys
import gc
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# inline magic registers the IPython PNG formatter for Figure objects;
# the KK panel builds raw Figures, which render only through that formatter
get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore[name-defined]
from pathlib import Path
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import load_ism, load_csv_spectrum, scan_input_spectra
from pipeline.quality import (
    run_linkk, select_best_replica, compute_frequency_cutoffs, kk_summary_table,
    strip_inductive,
)
from pipeline.plots import apply_pub_style

apply_pub_style()

sample_dir   = NOTEBOOK_DIR / sample_id
results_base = sample_dir / "Results"

# detect entry mode: input_spectra/ (CSV/TXT) or normal stage1_labeling.xlsx
_df_csv = scan_input_spectra(sample_dir)
_CSV_MODE = _df_csv is not None

if _CSV_MODE:
    print(f"Entry mode: input_spectra (CSV/TXT)")

conditions = get_selected_conditions()
if SKIP_EXISTING:
    skipped    = [c for c in conditions if (results_base / c / "stage2_kk.xlsx").exists()]
    conditions = [c for c in conditions if not (results_base / c / "stage2_kk.xlsx").exists()]
    if skipped:
        print(f"SKIP_EXISTING=True: skipping {len(skipped)} already computed condition(s)")

_LABELED_RE = re.compile(r'_\d{2,4}[Cc](?:_\d+)?\.[ci]', re.IGNORECASE)


def _short_cond(name: str) -> str:
    stripped = name[len(sample_id):].lstrip("_") if name.startswith(sample_id) else name
    parts = stripped.split("_")
    if parts and len(parts[0]) <= 3 and not re.match(r"^(Ar|O2|N2|H2)(?![A-Za-z0-9])", parts[0], re.I):
        stripped = "_".join(parts[1:])
    parts = stripped.split("_")
    if len(parts) >= 4 and parts[-3].isdigit() and parts[-2].isdigit():
        t_hi, t_lo = parts[-3], parts[-2]
        gas = " ".join(parts[:-3])
        return f"{gas} | {t_lo}-{t_hi}C"
    return stripped


n_ism_total = 0
if not _CSV_MODE:
    for c in conditions:
        xlsx_path = results_base / c / "stage1_labeling.xlsx"
        if xlsx_path.exists():
            try:
                df_tmp = pd.read_excel(xlsx_path, sheet_name="VALID")
                n_ism_total += df_tmp["file"].apply(lambda f: bool(_LABELED_RE.search(str(f)))).sum()
            except Exception as e:
                print(f"[WARN] Could not read {xlsx_path}: {e}. Check the file or re-run stage1_labeling.ipynb.")
else:
    n_ism_total = len(_df_csv[_df_csv["condition"].isin(conditions)])

print(f"Sample     : {sample_id}")
print(f"Conditions : {len(conditions)}")
for c in conditions:
    print(f"  {_short_cond(c)}")
print(f"Estimated  : {n_ism_total} spectra x ~20 ms = {n_ism_total * 0.020:.0f} s")

_STATUS_ICONS = {"GREEN": "✓ GREEN", "YELLOW": "⚠ YELLOW", "RED": "✗ RED"}


## Step 1: Batch KK

Runs Lin-KK on every (condition, T) and classifies each spectrum GREEN / YELLOW / RED.
If any YELLOW or RED appear: the Step 2 panel suggests `KK_OVERRIDES`; click Apply there, then re-run this cell.
When all GREEN: skip Step 2 and go directly to Step 3.

In [ ]:
# Cell A; silent batch run + classification table

def _resolve_cutoffs(condition: str, T_int: int) -> tuple:
    cond_ov = KK_OVERRIDES.get(condition, {})
    t_ov    = cond_ov.get(T_int, {})
    f_min_h = t_ov.get("f_min_hard", cond_ov.get("f_min_hard", KK_F_MIN_HARD))
    f_max_h = t_ov.get("f_max_hard", cond_ov.get("f_max_hard", KK_F_MAX_HARD))
    return f_min_h, f_max_h


def _n_in_hard_window(freq_b: np.ndarray, f_min_h, f_max_h) -> int:
    lo = f_min_h if f_min_h is not None else float(freq_b.min())
    hi = f_max_h if f_max_h is not None else float(freq_b.max())
    n  = int(((freq_b >= lo) & (freq_b <= hi)).sum())
    return n if n > 0 else len(freq_b)


def _classify_kk(kk_score: float, n_kept: int, n_total: int,
                 W_re: float | None = None, W_im: float | None = None) -> str:
    frac_cut = 1.0 - (n_kept / n_total) if n_total > 0 else 1.0
    if KK_USE_W_CRITERIA and (W_re is not None) and (W_im is not None):
        if W_re >= 0.95 and W_im >= 0.93 and frac_cut <= 0.20:
            return "GREEN"
        if W_re >= 0.90 and W_im >= 0.88 and frac_cut <= 0.40:
            return "YELLOW"
        return "RED"
    if kk_score >= 0.97 and frac_cut <= 0.20:
        return "GREEN"
    if kk_score >= 0.90 and frac_cut <= 0.40:
        return "YELLOW"
    return "RED"


def run_batch(quiet: bool = False):
    """Lin-KK on the selected conditions (and FOCUS_T); returns (styler, summary).

    Rebuilds all_kk_data / _kk_class / df_summary in place.
    """
    global all_kk_data, _kk_class, df_summary
    if FOCUS_T is not None:
        print(f"FOCUS_T = {FOCUS_T} C: processing only T={FOCUS_T}C across all conditions")

    all_kk_data = {}
    _kk_class   = {}

    _iter = conditions if quiet else tqdm(conditions, desc="Conditions", unit="cond")
    for condition in _iter:

        if _CSV_MODE:
            df_cond   = _df_csv[_df_csv["condition"] == condition].copy()
            t_groups  = sorted(df_cond["T_nominal"].dropna().unique())
            input_dir = sample_dir / "input_spectra" / condition
        else:
            xlsx_path     = sample_dir / "Results" / condition / "stage1_labeling.xlsx"
            df_stage1_all = pd.read_excel(xlsx_path, sheet_name="VALID")
            from pipeline.matching import _LABELED_RE as _RE
            labeled_mask  = df_stage1_all["file"].apply(lambda f: bool(_RE.search(str(f))))
            df_cond       = df_stage1_all[labeled_mask].copy()
            t_groups      = sorted(df_cond["T_nominal"].dropna().unique())
            input_dir     = sample_dir / "ISM validation" / condition

        cond_data  = {}
        cond_class = {}

        for T in t_groups:
            T_int = int(T)
            if FOCUS_T is not None and T_int != FOCUS_T:
                continue

            f_min_h, f_max_h = _resolve_cutoffs(condition, T_int)
            group_df         = df_cond[df_cond["T_nominal"] == T].sort_values("replica")

            records = []
            for _, row in group_df.iterrows():
                # Prefer the locally reconstructed path; the stored absolute
                # full_path is provenance metadata and may belong to another
                # machine or user (where exists() can raise PermissionError).
                fpath = input_dir / row["file"]
                if not fpath.exists() and pd.notna(row.get("full_path")):
                    try:
                        alt = Path(row["full_path"])
                        if alt.exists():
                            fpath = alt
                    except OSError:
                        pass
                if not fpath.exists():
                    continue
                if _CSV_MODE:
                    rec = load_csv_spectrum(fpath)
                else:
                    rec = load_ism(fpath)
                rec.T_nominal = T
                rec.T_mean    = row.get("T_mean")
                rec.pO2_mean  = row.get("pO2_mean")
                rec.replica   = row.get("replica")
                records.append(rec)

            if not records:
                continue

            kk_results = []
            n_inds = []
            for rec in records:
                freq_c, Z_re_c, Z_im_c, n_ind = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
                n_inds.append(n_ind)
                res = run_linkk(
                    freq_c, Z_re_c, Z_im_c,
                    c=KK_C, use_binary_M=KK_USE_BINARY_M, mu_target=KK_MU_TARGET,
                    iqr_fence_factor=KK_IQR_FENCE, iqr_window=KK_IQR_WINDOW,
                    f_min_hard=f_min_h, f_max_hard=f_max_h,
                )
                kk_results.append(res)

            best_idx = select_best_replica(kk_results)
            override_val = (OVERRIDES.get(condition, {}).get(T_int) or
                            OVERRIDES.get(condition, {}).get(T))
            if override_val:
                names = [r.path.name for r in records]
                if isinstance(override_val, list):
                    compare_idx = [names.index(f) for f in override_val if f in names]
                    if compare_idx:
                        best_idx = max(compare_idx, key=lambda i: kk_results[i]["kk_score"])
                elif override_val in names:
                    best_idx = names.index(override_val)

            best     = kk_results[best_idx]
            freq_b   = best["freq"]
            n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
            n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
            cls      = _classify_kk(best["kk_score"], n_kept, n_window,
                                    W_re=best["W_re"], W_im=best["W_im"])

            cond_data[T_int] = {
                "records":       records,
                "kk_results":    kk_results,
                "best_idx":      best_idx,
                "selected_file": records[best_idx].path.name,
                "n_stripped":    n_inds[best_idx],
                "f_min_cut":     best["f_min_cut"],
                "f_max_cut":     best["f_max_cut"],
                "df_summary":    kk_summary_table(records, kk_results, best_idx),
            }
            cond_class[T_int] = cls

        all_kk_data[condition] = cond_data
        _kk_class[condition]   = cond_class
        gc.collect()


    _STATUS_COLORS = {"GREEN": "#d4edda", "YELLOW": "#fff3cd", "RED": "#f8d7da"}


    def _build_summary_table():
        rows = []
        for condition in conditions:
            for T_int in sorted(_kk_class.get(condition, {}).keys(), reverse=True):
                data     = all_kk_data[condition][T_int]
                best     = data["kk_results"][data["best_idx"]]
                freq_b   = best["freq"]
                f_min_h, f_max_h = _resolve_cutoffs(condition, T_int)
                n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
                n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
                frac_cut_pct = round((1.0 - n_kept / n_window) * 100, 1) if n_window else 100.0
                rows.append({
                    "condition":  _short_cond(condition),
                    "T [°C]":     T_int,
                    "file":       data["selected_file"],
                    "n_ind":      data.get("n_stripped", 0),
                    "kk_score":   round(best["kk_score"], 3),
                    "W_re":       round(best["W_re"], 3),
                    "W_im":       round(best["W_im"], 3),
                    "f_min [Hz]": (round(best["f_min_cut"], 1)
                                   if best["f_min_cut"] is not None else None),
                    "f_max [Hz]": (round(best["f_max_cut"], 1)
                                   if best["f_max_cut"] is not None else None),
                    "% cut":      frac_cut_pct,
                    "STATUS":     _STATUS_ICONS[_kk_class[condition][T_int]],
                })
        return pd.DataFrame(rows)


    def _hl_status(val):
        for k, lbl in _STATUS_ICONS.items():
            if val == lbl:
                return f"background-color: {_STATUS_COLORS[k]}"
        return ""


    df_summary = _build_summary_table()
    styler = (
        df_summary.style
        .map(_hl_status, subset=["STATUS"])
        .format({"kk_score": "{:.3f}", "W_re": "{:.3f}", "W_im": "{:.3f}", "% cut": "{:.1f}"})
        .hide(axis="index")
    )

    n_green  = sum(v == "GREEN"  for d in _kk_class.values() for v in d.values())
    n_yellow = sum(v == "YELLOW" for d in _kk_class.values() for v in d.values())
    n_red    = sum(v == "RED"    for d in _kk_class.values() for v in d.values())
    n_total  = n_green + n_yellow + n_red
    t_label  = f" (FOCUS_T={FOCUS_T}C)" if FOCUS_T is not None else ""
    crit_lbl = "W-criteria (ceramic electrolyte)" if KK_USE_W_CRITERIA else "kk_score>=0.97 (strict)"
    summary = f"{n_total} spectra{t_label}: {n_green} GREEN | {n_yellow} YELLOW | {n_red} RED  [{crit_lbl}]"
    summary += ("\nCheck the suggested KK_OVERRIDES in the next cell, then run Step 2."
                if n_yellow + n_red > 0 else
                "\nAll clean. Skip Step 2 and go straight to Step 3 (export).")
    return styler, summary


_styler, _summary_txt = run_batch()
display(_styler)
print(_summary_txt)

In [ ]:
# Cell B; suggested KK_OVERRIDES (auto-generated from flagged spectra above)

_inv = {v: k for k, v in _STATUS_ICONS.items()}

needs = {
    cond: {
        T_int: {
            "f_min_hard": round(float(
                all_kk_data[cond][T_int]["kk_results"][
                    all_kk_data[cond][T_int]["best_idx"]
                ]["f_min_cut"]), 2),
            "f_max_hard": round(float(
                all_kk_data[cond][T_int]["kk_results"][
                    all_kk_data[cond][T_int]["best_idx"]
                ]["f_max_cut"]), 2),
        }
        for T_int, cls in T_dict.items()
        if cls in ("YELLOW", "RED")
    }
    for cond, T_dict in _kk_class.items()
    if any(cls in ("YELLOW", "RED") for cls in T_dict.values())
}

if not needs:
    print("All spectra GREEN; no overrides needed. Go to Step 3.")
else:
    print("# Suggested KK_OVERRIDES; paste into Step 2 below, adjust if needed")
    print("KK_OVERRIDES = {")
    for cond, T_dict in needs.items():
        print(f'    "{cond}": {{')
        for T_int, ov in sorted(T_dict.items(), reverse=True):
            tag = _kk_class[cond][T_int]
            print(f'        {T_int}: {{"f_min_hard": {ov["f_min_hard"]}, '
                  f'"f_max_hard": {ov["f_max_hard"]}}},  # {tag}')
        print("    },")
    print("}")

    # One-click apply (no copy-paste): merge suggestions into live KK_OVERRIDES.
    try:
        import ipywidgets as _W
        from IPython.display import display as _disp
        _btn = _W.Button(description="📥 Apply suggested overrides", button_style="warning",
                         layout=_W.Layout(width="300px"),
                         tooltip="Merge the suggestions above into the in-memory KK_OVERRIDES, then re-run Step 1")
        _msg = _W.HTML()
        def _apply_kk(_b):
            if PARAM_MODE == "lock":
                _msg.value = f"<b style='color:#9a6700'>{_LOCKED_MSG}</b>"
                return
            n = 0
            for _c, _td in needs.items():
                KK_OVERRIDES.setdefault(_c, {}).update(_td); n += len(_td)
            _save_overrides()
            _msg.value = (f"<b style='color:#b36b00'>Applied {n} override(s)</b> to KK_OVERRIDES, "
                          "re-run Step 1 (batch KK) to use them.")
        _btn.on_click(_apply_kk)
        _disp(_W.VBox([_btn, _msg]))
    except Exception as _e:
        print(f"[INFO] apply button needs ipywidgets ({_e}).")

## Step 2: Tune flagged spectra

Use the panel below. Dropdowns open on the worst-status spectrum first.
Move `f_min` / `f_max` until the residuals flatten, press **Preview this spectrum**
to recompute and redraw, then **Save f_min/f_max** for the chosen target
(one temperature, the whole condition, one temperature everywhere, or every
spectrum). **Delete saved f_min/f_max** reverts the same target to the
configuration values. Use Replica to compare or force a specific file.

Counters (✓ ⚠ ✗) update in real time; to refresh the Step 1 color table,
re-run Step 1 after finishing the tuning.


In [ ]:
# Quick KK tuning panel; re-run Lin-KK for one (condition, T) with custom f_min / f_max.
import io as _io
from matplotlib.figure import Figure
# Text and figure render into value-replaced widgets (W.HTML / W.Image):
# never a shared Output, whose lazy clear stacks duplicates in VSCode.
try:
    import ipywidgets as W
    from IPython.display import display as _display
    _HAS_WIDGETS_NB02 = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); KK tuning panel disabled.")
    _HAS_WIDGETS_NB02 = False


def _reclassify_all() -> None:
    global _kk_class
    for cond, td in all_kk_data.items():
        for T_int, data in td.items():
            best     = data["kk_results"][data["best_idx"]]
            freq_b   = best["freq"]
            f_min_h, f_max_h = _resolve_cutoffs(cond, T_int)
            n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
            n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
            cls      = _classify_kk(best["kk_score"], n_kept, n_window,
                                    W_re=best["W_re"], W_im=best["W_im"])
            _kk_class.setdefault(cond, {})[T_int] = cls


def _count_status() -> tuple[int, int, int]:
    g = sum(v == "GREEN"  for d in _kk_class.values() for v in d.values())
    y = sum(v == "YELLOW" for d in _kk_class.values() for v in d.values())
    r = sum(v == "RED"    for d in _kk_class.values() for v in d.values())
    return g, y, r


def _status_chips_html() -> str:
    g, y, r = _count_status()
    crit = "W-criteria (ceramic electrolyte)" if KK_USE_W_CRITERIA else "kk_score>=0.97 (strict)"
    return (f"<div style='font-size:13px; padding:4px 0'>"
            f"<span style='background:#d4edda; padding:3px 8px; border-radius:4px'>✓ {g}</span>&nbsp; "
            f"<span style='background:#fff3cd; padding:3px 8px; border-radius:4px'>⚠ {y}</span>&nbsp; "
            f"<span style='background:#f8d7da; padding:3px 8px; border-radius:4px'>✗ {r}</span>&nbsp;&nbsp; "
            f"<i>criterion: {crit}</i></div>")


_ST_ORDER = {"RED": 0, "YELLOW": 1, "GREEN": 2}
_ST_ICON  = {"RED": "✗", "YELLOW": "⚠", "GREEN": "✓"}


def _cond_worst(cond: str) -> str:
    vals = list(_kk_class.get(cond, {}).values())
    if "RED"    in vals: return "RED"
    if "YELLOW" in vals: return "YELLOW"
    return "GREEN"


def _cond_options() -> list:
    # fixed order from conditions list; icons update on retest but position never changes
    conds = [c for c in conditions if c in all_kk_data and all_kk_data[c]]
    return [(f"{_ST_ICON[_cond_worst(c)]}  {_short_cond(c)}", c) for c in conds]


def _T_options(cond: str) -> list:
    # fixed descending order 600→400; icons update on retest but position never changes
    ts = sorted(all_kk_data.get(cond, {}).keys(), reverse=True)
    return [(f"{_ST_ICON[_kk_class.get(cond, {}).get(t, 'GREEN')]}  {t} °C", t) for t in ts]


def _replica_options(cond: str, T: int) -> list:
    data = all_kk_data.get(cond, {}).get(T)
    if not data:
        return [("(no data)", None)]
    best_idx = data["best_idx"]
    forced   = OVERRIDES.get(cond, {}).get(T)
    # The KK-best replica stays marked even when an override is active,
    # so the automatic choice is always recoverable at a glance.
    opts = []
    for i, rec in enumerate(data["records"]):
        tags = []
        if i == best_idx:
            tags.append("KK best" if forced else "auto")
        if forced and rec.path.name == forced:
            tags.append("forced")
        label = rec.path.name + (f"  ({', '.join(tags)})" if tags else "")
        opts.append((label, rec.path.name))
    return opts


def _retest_kk(condition: str, T_int: int, f_min: float | None, f_max: float | None,
               fence: float | None = None, window: int | None = None) -> tuple:
    """Re-run Lin-KK for one (condition, T); return (message, Figure or None)."""
    data = all_kk_data.get(condition, {}).get(T_int)
    if data is None:
        return (f"[WARN] no KK data for {condition} T={T_int}; run Step 1 first.", None)
    rec = data["records"][data["best_idx"]]
    # Preview must honour a forced replica exactly like the batch cell.
    _forced = OVERRIDES.get(condition, {}).get(T_int)
    _names  = [r.path.name for r in data["records"]]
    if isinstance(_forced, list):
        _idxs = [_names.index(f) for f in _forced if f in _names]
        if _idxs:
            rec = data["records"][max(_idxs, key=lambda i: data["kk_results"][i]["kk_score"])]
    elif _forced in _names:
        rec = data["records"][_names.index(_forced)]

    msg_lines = []
    if (f_min is not None or f_max is not None) and PARAM_MODE != "lock":
        KK_OVERRIDES.setdefault(condition, {}).setdefault(T_int, {})
        if f_min is not None: KK_OVERRIDES[condition][T_int]["f_min_hard"] = f_min
        if f_max is not None: KK_OVERRIDES[condition][T_int]["f_max_hard"] = f_max
        _save_overrides()
    elif PARAM_MODE == "lock":
        msg_lines.append(f"viewing only, {_LOCKED_MSG}")

    freq_c, Z_re_c, Z_im_c, n_ind = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
    res = run_linkk(
        freq_c, Z_re_c, Z_im_c,
        c=KK_C, use_binary_M=KK_USE_BINARY_M, mu_target=KK_MU_TARGET,
        iqr_fence_factor=(fence if fence is not None else KK_IQR_FENCE),
        iqr_window=(window if window is not None else KK_IQR_WINDOW),
        f_min_hard=f_min, f_max_hard=f_max,
    )
    freq_b = res["freq"]
    n_win  = _n_in_hard_window(freq_b, f_min, f_max)
    n_kept = int(((freq_b >= res["f_min_cut"]) & (freq_b <= res["f_max_cut"])).sum())
    cls    = _classify_kk(res["kk_score"], n_kept, n_win, W_re=res["W_re"], W_im=res["W_im"])
    msg_lines.append(
        f"file={rec.path.name}  kk={res['kk_score']:.4f}  "
        f"W_re={res['W_re']:.3f}  W_im={res['W_im']:.3f}  "
        f"M={res['M']}  mu={res['mu']:.2f}  [{cls}]  "
        f"f_min_cut={res['f_min_cut']:.1f} Hz  f_max_cut={res['f_max_cut']:.1f} Hz")

    fq       = res["freq"]
    rr       = res["res_re"] * 100
    ri       = res["res_im"] * 100
    fence_pc = res["cutoff_fence"] * 100
    lo, hi   = res["f_min_cut"], res["f_max_cut"]

    _o   = np.argsort(freq_c)
    frs  = freq_c[_o]; zres = Z_re_c[_o]; zims = Z_im_c[_o]
    kept = (frs >= lo) & (frs <= hi)
    Zmag  = np.sqrt(zres**2 + zims**2) / 1e3
    phase = np.degrees(np.arctan2(-zims, zres))

    # raw Figure (not plt.figure): exception-safe, cannot leak into other cells
    fig = Figure(figsize=(11.5, 5.9))
    gs  = fig.add_gridspec(2, 2, width_ratios=[1.15, 1.0], height_ratios=[1, 1],
                           hspace=0.42, wspace=0.26)
    axB = fig.add_subplot(gs[0, 0])
    axR = fig.add_subplot(gs[1, 0], sharex=axB)
    axN = fig.add_subplot(gs[:, 1])
    _band = dict(color="#2ecc71", alpha=0.10)
    def _cut_lines(ax):
        if lo: ax.axvline(lo, color="grey", ls="--", lw=1.0)
        if hi: ax.axvline(hi, color="grey", ls="-.", lw=1.0)

    axB.loglog(frs, Zmag, "o-", ms=3, lw=0.8, color="#333")
    axB.set_ylabel(r"$|Z|$ / k$\Omega$")
    axBp = axB.twinx()
    axBp.semilogx(frs, phase, "s-", ms=2.6, lw=0.7, color="#9b59b6")
    axBp.set_ylabel("phase / deg", color="#9b59b6")
    axBp.tick_params(axis="y", labelcolor="#9b59b6")
    axB.axvspan(lo, hi, **_band); _cut_lines(axB); axB.set_title("Bode")
    plt.setp(axB.get_xticklabels(), visible=False)

    axR.semilogx(fq, rr, "o-", ms=3, lw=0.8, color="#2471a3", label="Re")
    axR.semilogx(fq, ri, "o-", ms=3, lw=0.8, color="#e67e22", label="Im")
    axR.axhline( fence_pc, color="steelblue", ls="--", lw=0.9, label=f"±{fence_pc:.1f}%")
    axR.axhline(-fence_pc, color="steelblue", ls="--", lw=0.9)
    axR.axhline(0, color="k", lw=0.4)
    axR.axvspan(lo, hi, **_band); _cut_lines(axR)
    axR.set_xlabel("f / Hz"); axR.set_ylabel(r"residual / $|Z|$  %")
    axR.set_title("Lin-KK residuals")
    axR.legend(fontsize=8, frameon=False, loc="upper left", ncol=3)

    zk_r, zk_i = zres[kept] / 1e3, zims[kept] / 1e3
    if zk_r.size:
        sc = axN.scatter(zk_r, zk_i, c=np.log10(frs[kept]), cmap="viridis",
                         s=26, zorder=3, linewidth=0)
        _m = max(zk_r.max(), zk_i.max()) * 1.08
        axN.set_xlim(0, _m); axN.set_ylim(0, _m)
        cb = fig.colorbar(sc, ax=axN, fraction=0.046, pad=0.04)
        cb.set_label(r"$\log_{10}(f\,/\,\mathrm{Hz})$")
    axN.set_aspect("equal", "box")
    axN.set_xlabel(r"$Z'$ / k$\Omega$"); axN.set_ylabel(r"$-Z''$ / k$\Omega$")
    axN.set_title("Nyquist  (color = frequency)")
    axN.text(0.04, 0.94, f"cut {(~kept).sum()} pts", transform=axN.transAxes,
             fontsize=8, color="grey")
    return "\n".join(msg_lines), fig


if _HAS_WIDGETS_NB02 and all_kk_data:
    _cond_opts = _cond_options()
    if _cond_opts:
        _c0_2   = _cond_opts[0][1]
        _T_opts = _T_options(_c0_2)
        _T0_2   = _T_opts[0][1] if _T_opts else 600

        wc  = W.Dropdown(options=_cond_opts, value=_c0_2, description="Cond:",
                         layout=W.Layout(width="420px"))
        wT  = W.Dropdown(options=_T_opts, value=_T0_2, description="T [C]:",
                         layout=W.Layout(width="220px"))
        wmn = W.FloatLogSlider(value=KK_F_MIN_HARD or 30, base=10, min=-2, max=6, step=0.02,
                               description="f_min Hz", readout_format=".2g",
                               continuous_update=False, layout=W.Layout(width="300px"))
        wmn_txt = W.BoundedFloatText(value=KK_F_MIN_HARD or 30, min=0.001, max=1e7, step=0.01,
                                     layout=W.Layout(width="110px"),
                                     tooltip="type exact f_min [Hz]")
        wmx = W.FloatLogSlider(value=KK_F_MAX_HARD or 1e6, base=10, min=0, max=8, step=0.02,
                               description="f_max Hz", readout_format=".1e",
                               continuous_update=False, layout=W.Layout(width="300px"))
        wmx_txt = W.BoundedFloatText(value=KK_F_MAX_HARD or 1e6, min=1, max=1e8, step=1,
                                     layout=W.Layout(width="110px"),
                                     tooltip="type exact f_max [Hz]")
        wfence = W.FloatSlider(value=KK_IQR_FENCE, min=0.1, max=3.0, step=0.1,
                               description="fence", readout_format=".1f",
                               continuous_update=False, layout=W.Layout(width="360px"),
                               tooltip="IQR fence multiplier: lower = stricter edge cut")
        wwin = W.IntSlider(value=KK_IQR_WINDOW, min=3, max=20, step=1,
                           description="window", continuous_update=False,
                           layout=W.Layout(width="300px"),
                           tooltip="consecutive clean points to confirm the cut edge")
        wgo = W.Button(description="Preview this spectrum", button_style="primary",
                       layout=W.Layout(width="190px"),
                       tooltip="Re-run Lin-KK for the selected (condition, T) and redraw")

        _w_replica = W.Dropdown(
            options=_replica_options(_c0_2, _T0_2),
            description="Replica:",
            layout=W.Layout(width="500px"),
            style={"description_width": "60px"}
        )
        _w_force = W.Button(
            description="Force this replica",
            layout=W.Layout(width="185px"),
            tooltip="Add to OVERRIDES; re-run Step 1 then Step 3 to export this file"
        )

        chips = W.HTML(value=_status_chips_html())
        txt2  = W.HTML()
        img2  = W.Image(format="png",
                        layout=W.Layout(width="100%", max_width="1265px"))

        def _say(msg: str) -> None:
            txt2.value = ("<pre style='font-size:12px;margin:4px 0'>"
                          + msg + "</pre>")

        _syncing = [False]
        _kk_suspend = [False]

        def _wmn_slider_to_txt(change):
            if not _syncing[0]:
                _syncing[0] = True
                try:
                    wmn_txt.value = round(change["new"], 2)
                finally:
                    _syncing[0] = False

        def _wmn_txt_to_slider(change):
            if not _syncing[0] and change["new"] > 0:
                _syncing[0] = True
                try:
                    wmn.value = change["new"]
                finally:
                    _syncing[0] = False

        def _wmx_slider_to_txt(change):
            if not _syncing[0]:
                _syncing[0] = True
                try:
                    wmx_txt.value = round(change["new"], 2)
                finally:
                    _syncing[0] = False

        def _wmx_txt_to_slider(change):
            if not _syncing[0] and change["new"] > 0:
                _syncing[0] = True
                try:
                    wmx.value = change["new"]
                finally:
                    _syncing[0] = False

        wmn.observe(_wmn_slider_to_txt, names="value")
        wmn_txt.observe(_wmn_txt_to_slider, names="value")
        wmx.observe(_wmx_slider_to_txt, names="value")
        wmx_txt.observe(_wmx_txt_to_slider, names="value")

        def _load_state_into_widgets(*_):
            _kk_suspend[0] = True
            cond, T = wc.value, int(wT.value)
            ov   = KK_OVERRIDES.get(cond, {})
            t_ov = ov.get(T, {}) if isinstance(ov.get(T), dict) else {}
            fmin = t_ov.get("f_min_hard",
                            ov.get("f_min_hard") if not isinstance(ov.get("f_min_hard"), dict) else None)
            fmax = t_ov.get("f_max_hard",
                            ov.get("f_max_hard") if not isinstance(ov.get("f_max_hard"), dict) else None)
            wmn.value = float(fmin) if fmin is not None else (KK_F_MIN_HARD or 30)
            wmx.value = float(fmax) if fmax is not None else (KK_F_MAX_HARD or 1e6)
            wmn_txt.value = round(wmn.value, 2)
            wmx_txt.value = round(wmx.value, 2)
            _kk_suspend[0] = False

        def _refresh_replica(*_):
            cond, T = wc.value, int(wT.value)
            opts = _replica_options(cond, T)
            cur = _w_replica.value
            _w_replica.options = opts
            vals = [v for _, v in opts]
            if cur in vals:
                _w_replica.value = cur
            elif opts and opts[0][1] is not None:
                _w_replica.value = opts[0][1]

        def _refresh_dropdowns(*_):
            cur_cond = wc.value
            new_cond_opts = _cond_options()
            cur_T = wT.value
            wc.options = new_cond_opts
            cond_vals = [v for _, v in new_cond_opts]
            if cur_cond in cond_vals:
                wc.value = cur_cond
            new_T_opts = _T_options(wc.value)
            wT.options = new_T_opts
            T_vals = [v for _, v in new_T_opts]
            if cur_T in T_vals:
                wT.value = cur_T

        def _refresh_T(*_):
            cond = wc.value
            if cond is None:
                return
            cur_T      = wT.value
            new_T_opts = _T_options(cond)
            wT.options = new_T_opts
            T_vals     = [v for _, v in new_T_opts]
            wT.value   = cur_T if cur_T in T_vals else (new_T_opts[0][1] if new_T_opts else None)
            _load_state_into_widgets()
            _refresh_replica()

        wc.observe(_refresh_T, names="value")
        wT.observe(_load_state_into_widgets, names="value")
        wT.observe(_refresh_replica, names="value")

        def _on_force_replica(_btn):
            _refresh_replica()   # resync to the current (cond, T) before reading the dropdown
            cond, T = wc.value, int(wT.value)
            fname = _w_replica.value
            if fname is None:
                return
            data = all_kk_data.get(cond, {}).get(T)
            if not data:
                return
            best_name = data["records"][data["best_idx"]].path.name
            if fname == best_name:
                # Merge-save can only add keys: clear the saved entry on disk too.
                removed = remove_override_entries(sample_id, "overrides", cond, T)
                sub = OVERRIDES.get(cond)
                if isinstance(sub, dict):
                    sub.pop(T, None)
                    if not sub:
                        OVERRIDES.pop(cond, None)
                if removed:
                    _say(f"Override removed for {_short_cond(cond)} T={T}; auto-selection restored.")
                else:
                    _say("Already auto-selected; no override stored.")
            else:
                OVERRIDES.setdefault(cond, {})[T] = fname
                _say(f"OVERRIDES['{cond}'][{T}] = '{fname}'\n"
                     "Re-run Step 1 (batch KK) then Step 3 (export) to apply.")
                _save_overrides()
            _refresh_replica()
            _ov_source_html()
            _on_retest()   # replot the preview with the replica now in effect
        _w_force.on_click(_on_force_replica)

        _busy2 = [False]

        def _on_retest(_btn=None):
            if _kk_suspend[0] or _busy2[0]:
                return
            _busy2[0] = True
            wgo.disabled = True
            try:
                global KK_IQR_FENCE, KK_IQR_WINDOW
                if PARAM_MODE != "lock":
                    KK_IQR_FENCE  = float(wfence.value)
                    KK_IQR_WINDOW = int(wwin.value)
                msg, fig = _retest_kk(wc.value, int(wT.value),
                                      float(wmn.value) if wmn.value > 0 else None,
                                      float(wmx.value) if wmx.value > 0 else None,
                                      fence=float(wfence.value), window=int(wwin.value))
                _reclassify_all()
                chips.value = _status_chips_html()
                _refresh_dropdowns()
                _ov_source_html()
                g, y, r = _count_status()
                tail = (f"-> {y} YELLOW, {r} RED remaining. Re-run Step 1 to refresh the table."
                        if y + r > 0 else
                        "-> All spectra GREEN. Re-run Step 1 to refresh the table, then go to Step 3.")
                _say(msg + "\n" + tail)
                if fig is not None:
                    _buf = _io.BytesIO()
                    fig.savefig(_buf, format="png", dpi=110)
                    img2.value = _buf.getvalue()
            finally:
                _busy2[0] = False
                wgo.disabled = False
        wgo.on_click(_on_retest)

        from pipeline.session import remove_override_entries

        ov_src = W.HTML()
        w_scope = W.Dropdown(
            options=[("only this temperature", "T"),
                     ("all temperatures of this condition", "cond"),
                     ("this temperature in every condition", "allcond"),
                     ("every spectrum", "all")],
            value="cond", description="Apply to:",
            layout=W.Layout(width="330px"), style={"description_width": "70px"})
        _w_rm = W.Button(description="🗑 Delete saved f_min/f_max",
                         layout=W.Layout(width="230px"),
                         tooltip="Delete the saved cutoffs (and forced replicas) for the "
                                 "chosen target; those spectra follow the config cell again")

        def _ov_source_html(*_):
            cond, T = wc.value, int(wT.value)
            t_ov = KK_OVERRIDES.get(cond, {}).get(T, {}) or {}
            parts = []
            for lbl, k, glob in (("f_min", "f_min_hard", KK_F_MIN_HARD),
                                 ("f_max", "f_max_hard", KK_F_MAX_HARD)):
                if k in t_ov:
                    parts.append(f"{lbl} = {t_ov[k]:g} Hz <b style='color:#b36b00'>[saved override]</b>")
                else:
                    v = "adaptive" if glob is None else f"{glob:g} Hz"
                    parts.append(f"{lbl} = {v} [config cell]")
            forced = OVERRIDES.get(cond, {}).get(T)
            if forced:
                parts.append(f"replica <b style='color:#b36b00'>[forced: {forced}]</b>")
            ov_src.value = "<span style='font-size:12px'>" + " &nbsp;|&nbsp; ".join(parts) + "</span>"

        w_ap = W.Button(description="💾 Save f_min/f_max", button_style="success",
                        layout=W.Layout(width="200px"),
                        tooltip="Save the current f_min/f_max for every "
                                "(condition, T) in the chosen target")

        def _scope_targets():
            cond, T = wc.value, int(wT.value)
            if w_scope.value == "T":
                return [(cond, T)]
            if w_scope.value == "cond":
                return [(cond, t) for t in all_kk_data.get(cond, {})]
            if w_scope.value == "allcond":
                return [(c, t) for c, td in all_kk_data.items() for t in td if t == T]
            return [(c, t) for c, td in all_kk_data.items() for t in td]

        def _on_apply_scoped(_btn):
            if PARAM_MODE == "lock":
                _say(_LOCKED_MSG)
                return
            f_min, f_max = float(wmn.value), float(wmx.value)
            targets = _scope_targets()
            for c, t in targets:
                KK_OVERRIDES.setdefault(c, {}).setdefault(t, {})
                KK_OVERRIDES[c][t]["f_min_hard"] = f_min
                KK_OVERRIDES[c][t]["f_max_hard"] = f_max
            _save_overrides()
            note = ""
            if FOCUS_T is not None:
                note = ("\nNote: the last batch ran with Focus T set, so the targets "
                        "cover only that subset; set Focus T to (all T) and re-run "
                        "Step 1 to reach everything.")
            _say(f"Saved f_min={f_min:g} Hz, f_max={f_max:g} Hz for "
                 f"{len(targets)} (condition, T) pair(s) from the last processed batch.\n"
                 "Re-run Step 1 (batch KK) to reclassify with these cutoffs." + note)
            _ov_source_html()
        w_ap.on_click(_on_apply_scoped)

        def _on_remove_override(_btn):
            # Deletion works on session.json directly, so it reaches entries
            # saved in earlier sessions even if the last batch was focused.
            if PARAM_MODE == "lock":
                _say(_LOCKED_MSG)
                return
            cond, T = wc.value, int(wT.value)
            _saved = load_sample(sample_id)
            _conds = sorted(set(list(_saved.get("kk_overrides", {}))
                                + list(_saved.get("overrides", {}))))
            if w_scope.value == "T":
                jobs = [(cond, T)]
            elif w_scope.value == "cond":
                jobs = [(cond, None)]
            elif w_scope.value == "allcond":
                jobs = [(c, T) for c in _conds]
            else:
                jobs = [(c, None) for c in _conds]
            n = 0
            for c, t in jobs:
                for key, store in (("kk_overrides", KK_OVERRIDES), ("overrides", OVERRIDES)):
                    if remove_override_entries(sample_id, key, c, t):
                        n += 1
                    if t is None:
                        store.pop(c, None)
                    else:
                        sub = store.get(c)
                        if isinstance(sub, dict):
                            sub.pop(t, None)
                            if not sub:
                                store.pop(c, None)
            if n:
                _say(f"Deleted the saved cutoffs/replicas for the chosen target "
                     f"({n} stored entr{'y' if n == 1 else 'ies'} removed from session.json).\n"
                     "Those spectra follow the config-cell values again; re-run Step 1 to refresh.")
            else:
                _say("No saved cutoffs stored for the chosen target.")
            _ov_source_html()
            _refresh_replica()
        _w_rm.on_click(_on_remove_override)

        # live update: re-test on slider release (safe: value-replaced image + busy guard)
        wmn.observe(lambda ch: _on_retest(), names="value")
        wmx.observe(lambda ch: _on_retest(), names="value")
        wc.observe(_ov_source_html, names="value")
        wT.observe(_ov_source_html, names="value")

        if PARAM_MODE == "lock":
            # reproduction mode: viewing allowed, state-changing controls off
            for _wl in (_w_force, _w_rm, w_ap):
                _wl.disabled = True
                _wl.tooltip  = _LOCKED_MSG
        _load_state_into_widgets()
        _refresh_replica()
        _ov_source_html()
        _display(W.VBox([
            W.HBox([wc, wT]),
            W.HBox([wmn, wmn_txt]),
            W.HBox([wmx, wmx_txt]),
            W.HBox([wfence, wwin, wgo]),
            W.HBox([_w_replica, _w_force]),
            ov_src,
            W.HBox([w_scope, w_ap, _w_rm]),
            chips, txt2, img2,
        ]))
elif not all_kk_data:
    print("[INFO] No KK data; run Step 1 first.")

## Step 3: Selection summary and export

Compact per-condition table of the selected replicas and writes `stage2_kk.xlsx`
(All + Selected sheets) consumed by Stage 3.

When `FOCUS_T` is set, the export **merges** into the existing file; rows for other
temperatures are preserved. `FOCUS_T = None` → full overwrite (safe for first run).

In [ ]:
# Summary Styler table per condition
from pipeline.utils import merge_sheet_by_T, build_metadata_sheet

for condition, cond_data in all_kk_data.items():
    if not cond_data:
        continue
    print(f"\nCondition: {condition}")

    rows = []
    for T_int in sorted(cond_data.keys(), reverse=True):
        data = cond_data[T_int]
        best = data["kk_results"][data["best_idx"]]
        rec  = data["records"][data["best_idx"]]

        if   best["pass_re"] and best["pass_im"]:     kk_lbl = "✓ PASS"
        elif best["pass_re"] and not best["pass_im"]: kk_lbl = "✗ Re only"
        elif best["pass_im"] and not best["pass_re"]: kk_lbl = "✗ Im only"
        else:                                         kk_lbl = "✗ fail"

        rows.append({
            "T [°C]":    T_int,
            "pO2 [bar]": round(float(rec.pO2_mean or 0), 4),
            "file":      data["selected_file"],
            "kk_score":  round(best["kk_score"], 3),
            "W_re":      round(best["W_re"], 3),
            "W_im":      round(best["W_im"], 3),
            "KK":        kk_lbl,
            "f_min [Hz]": (round(data["f_min_cut"], 1)
                           if data["f_min_cut"] is not None else "-"),
            "f_max [Hz]": (round(data["f_max_cut"], 1)
                           if data["f_max_cut"] is not None else "-"),
            "★": "★",
        })

    df_disp = pd.DataFrame(rows)

    def _highlight_kk(val):
        return "background-color: #fff3cd" if "✗" in str(val) else ""

    styler = (
        df_disp.style
        .map(_highlight_kk, subset=["KK"])
        .format({"kk_score": "{:.3f}", "W_re": "{:.3f}", "W_im": "{:.3f}"})
        .set_caption(condition.replace("_", " "))
        .set_table_styles([{
            "selector": "caption",
            "props": "font-size: 11px; font-weight: bold; text-align: left;"
        }])
        .hide(axis="index")
    )
    display(styler)

# Build Metadata DataFrame (Lin-KK fixed parameters; applied to all conditions)
df_meta = build_metadata_sheet(
    sample_id  = sample_id,
    stage_name = "stage2_kk",
    params = {
        "KK_C":              KK_C,
        "KK_MU_TARGET":      KK_MU_TARGET,
        "KK_USE_BINARY_M":   KK_USE_BINARY_M,
        "KK_IQR_FENCE":      KK_IQR_FENCE,
        "KK_IQR_WINDOW":     KK_IQR_WINDOW,
        "KK_F_MIN_HARD":     KK_F_MIN_HARD,
        "KK_F_MAX_HARD":     KK_F_MAX_HARD,
        "acceptance_GREEN":  "kk_score >= 0.97 AND frac_cut <= 0.20",
        "acceptance_YELLOW": "kk_score >= 0.90 AND frac_cut <= 0.40",
        "reference":         "Schoenleber et al., Electrochim. Acta 131 (2014); RelaxIS manual v1.31",
    },
)

# Export stage2_kk.xlsx (merge-aware when FOCUS_T is set)
print("\nExporting...")
for condition, cond_data in all_kk_data.items():
    if not cond_data:
        print(f"  [{condition}] No data; skipped.")
        continue

    results_dir = sample_dir / "Results" / condition
    results_dir.mkdir(parents=True, exist_ok=True)
    rows_all, rows_sel = [], []

    for T_int, data in sorted(cond_data.items()):
        for i, (rec, res) in enumerate(zip(data["records"], data["kk_results"])):
            f_min, f_max = compute_frequency_cutoffs(res)
            is_sel = i == data["best_idx"]
            row = {
                "condition": condition, "file": rec.path.name,
                "full_path": str(rec.path), "T_nominal": T_int,
                "T_mean":    round(rec.T_mean, 2) if rec.T_mean is not None else None,
                "pO2_mean":  rec.pO2_mean, "replica": rec.replica,
                "kk_score":  round(res["kk_score"], 4),
                "W_re":      round(res["W_re"], 4),
                "W_im":      round(res["W_im"], 4),
                "pass_re":   res["pass_re"],
                "pass_im":   res["pass_im"],
                "mu":        round(res.get("mu", float("nan")), 3),
                "M":         res.get("M"),
                "max_res_re":    round(float(np.abs(res["res_re"]).max()), 4),
                "max_res_im":    round(float(np.abs(res["res_im"]).max()), 4),
                "cutoff_fence":  round(res.get("cutoff_fence", float("nan")), 4),
                "f_min_cut": f_min, "f_max_cut": f_max, "selected": is_sel,
            }
            rows_all.append(row)
            if is_sel:
                rows_sel.append(row)

    xlsx_path = results_dir / "stage2_kk.xlsx"

    df_all = merge_sheet_by_T(xlsx_path, "All",      pd.DataFrame(rows_all), FOCUS_T)
    df_sel = merge_sheet_by_T(xlsx_path, "Selected", pd.DataFrame(rows_sel), FOCUS_T)
    _export_mode = f"merged T={FOCUS_T}°C" if (FOCUS_T is not None and xlsx_path.exists()) else "full overwrite"

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df_all.to_excel(writer,  sheet_name="All",      index=False)
        df_sel.to_excel(writer,  sheet_name="Selected", index=False)
        df_meta.to_excel(writer, sheet_name="Metadata", index=False)

    print(f"  [{condition}]  {len(df_sel)} selected → {xlsx_path.relative_to(NOTEBOOK_DIR)}  [{_export_mode}]")

print("\nExport complete.")
print("→ Next: stage3_drt.ipynb")

In [ ]:
if _save_overrides():
    print("Saved:", list(KK_OVERRIDES.keys()))
else:
    print(_LOCKED_MSG)

**Next step:** run [stage3_drt.ipynb](stage3_drt.ipynb)